## 퍼널 분석 (Funnel Analysis)
- 목적: 주문~리뷰까지 각 단계별 소요시간 측정, 병목 구간 탐지
- 핵심 질문: 판매자가 택배사에 늦게 준 것인가, 택배사가 배송을 오래 한 것인가?
- 사용 컬럼: order_purchase_timestamp · order_approved_at · order_delivered_carrier_date · order_delivered_customer_date
- 시각화: 구간별 소요시간 히스토그램 · 박스플롯

## 코호트 분석 (Cohort Analysis)
- 목적: 배송 지연 경험 고객의 재구매율 추적
- 가설: 배송 지연 경험 고객은 정상 배송 고객보다 리텐션이 낮을 것이다
- 사용 컬럼: customer_unique_id · purchase_month · is_late · review_score
- 시각화: 재구매율 히트맵 · 그룹별 리텐션 곡선

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATE_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df = pd.read_csv("funnel_df.csv", parse_dates=DATE_COLS)
valid = df[df['is_valid_funnel']].copy()

print('전체:', df.shape)
print('유효 주문:', valid.shape)
print()
print(valid[['t_approve_d','t_carrier_d','t_delivery_d']].describe().round(2))

전체: (99441, 18)
유효 주문: (95088, 18)

       t_approve_d  t_carrier_d  t_delivery_d
count     95088.00     95088.00      95088.00
mean          0.40         2.85          9.36
std           0.80         3.48          8.77
min           0.00         0.00          0.00
25%           0.01         0.90          4.11
50%           0.01         1.85          7.11
75%           0.56         3.62         12.06
max          30.89       125.76        205.19


In [6]:
# 1. order_status 확인
print('is_valid_funnel vs delivered 비교')
print('is_valid_funnel True:', df['is_valid_funnel'].sum())
print('delivered 건수:', (df['order_status'] == 'delivered').sum())
print()

# 2. valid 기준으로 확정
valid = df[df['is_valid_funnel']].copy()

# 3. review_score 결측 확인
print('review_score 결측:', valid['review_score'].isnull().sum())

# 4. outlier 확인
print()
print('t_carrier_d 99퍼센타일:', valid['t_carrier_d'].quantile(0.99).round(2))
print('t_delivery_d 99퍼센타일:', valid['t_delivery_d'].quantile(0.99).round(2))

is_valid_funnel vs delivered 비교
is_valid_funnel True: 95088
delivered 건수: 96478

review_score 결측: 639

t_carrier_d 99퍼센타일: 17.14
t_delivery_d 99퍼센타일: 41.03


In [7]:
# =============================================
# EDA - 배송 지연 → 만족도 → 재구매율 연결
# =============================================

# 1. 전체 지연율
print('=== 1. 전체 지연율 ===')
print(f"전체 유효 주문: {len(valid):,}건")
print(f"지연 주문: {valid['is_delayed'].sum():,}건")
print(f"지연율: {valid['is_delayed'].mean()*100:.1f}%")

# 2. 지연 여부별 리뷰 점수 비교
print()
print('=== 2. 지연 여부별 리뷰 점수 ===')
review_by_delay = valid.groupby('is_delayed')['review_score'].agg(['mean','median','count'])
review_by_delay.index = ['정상 배송', '지연 배송']
print(review_by_delay.round(2))

=== 1. 전체 지연율 ===
전체 유효 주문: 95,088건
지연 주문: 7,793건
지연율: 8.2%

=== 2. 지연 여부별 리뷰 점수 ===
       mean  median  count
정상 배송  4.29     5.0  86821
지연 배송  2.57     2.0   7628


In [8]:
# 3. 지연 여부별 재구매율
print('=== 3. 지연 여부별 재구매율 ===')
purchase_count = valid.groupby('customer_unique_id')['order_id'].count().reset_index()
purchase_count.columns = ['customer_unique_id', 'order_count']

first_order = valid.sort_values('order_purchase_timestamp').drop_duplicates('customer_unique_id')[['customer_unique_id','is_delayed']]
first_order = first_order.merge(purchase_count, on='customer_unique_id')
first_order['is_repurchase'] = (first_order['order_count'] >= 2).astype(int)

repurchase = first_order.groupby('is_delayed')['is_repurchase'].mean() * 100
repurchase.index = ['정상 배송', '지연 배송']
print(repurchase.round(2))

=== 3. 지연 여부별 재구매율 ===
정상 배송    3.03
지연 배송    2.47
Name: is_repurchase, dtype: float64


In [9]:
print('=== 전체 재구매율 ===')
print(f"전체 고객수: {len(purchase_count):,}명")
print(f"재구매 고객수: {(purchase_count['order_count'] >= 2).sum():,}명")
print(f"전체 재구매율: {(purchase_count['order_count'] >= 2).mean()*100:.2f}%")

=== 전체 재구매율 ===
전체 고객수: 92,031명
재구매 고객수: 2,744명
전체 재구매율: 2.98%


In [10]:
# =============================================
# EDA - 배송 소요일 vs 리뷰 점수 상관관계
# =============================================

# 1. 배송 소요일 구간별 리뷰 점수
print('=== 배송 소요일 구간별 리뷰 점수 ===')
valid['delivery_bucket'] = pd.cut(
    valid['t_total_d'],
    bins=[0, 5, 10, 15, 25, 999],
    labels=['5일 이하', '6~10일', '11~15일', '16~25일', '25일 초과']
)

bucket_review = valid.groupby('delivery_bucket')['review_score'].agg(['mean','count'])
print(bucket_review.round(2))

print()

# 2. 배송 소요일 vs 재구매율
print('=== 배송 소요일 구간별 재구매율 ===')
valid_with_repurchase = valid.merge(
    first_order[['customer_unique_id','is_repurchase']], 
    on='customer_unique_id', how='left'
)

bucket_repurchase = valid_with_repurchase.groupby('delivery_bucket')['is_repurchase'].mean() * 100
print(bucket_repurchase.round(2))

=== 배송 소요일 구간별 리뷰 점수 ===
                 mean  count
delivery_bucket             
5일 이하            4.45  12996
6~10일            4.36  32264
11~15일           4.27  23241
16~25일           4.04  18338
25일 초과           2.70   7610

=== 배송 소요일 구간별 재구매율 ===
delivery_bucket
5일 이하     6.02
6~10일     6.02
11~15일    6.37
16~25일    6.26
25일 초과    5.41
Name: is_repurchase, dtype: float64
